# Imperial BBO Capstone: final reproducibility notebook

This notebook provides a short assessor-facing route through the complete recorded evidence. It verifies the 175 starter observations and 104 prospective portal evaluations, compares starter and participant-query results, and checks the final winner summary. The hidden Imperial functions are external, so the notebook reproduces the analysis rather than the undisclosed objective evaluations.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').is_file() and (candidate / 'BBO_Dashboard/data/complete_internal_evidence.csv').is_file():
            return candidate
    raise FileNotFoundError('Repository files were not found. Open this notebook from anywhere inside the cloned Imperial_BBO_Capstone repository.')

ROOT = find_repository_root(Path.cwd())

DATA = ROOT / 'BBO_Dashboard/data/complete_internal_evidence.csv'
WINNERS = ROOT / 'Module_25_Final_BBO_Submission/Final_13_Round_Evidence/FINAL_RESULTS_SUMMARY.csv'
evidence = pd.read_csv(DATA)
winners = pd.read_csv(WINNERS)
print(f'Repository root: {ROOT}')
print(f'Complete evidence shape: {evidence.shape}')


In [ ]:
source_counts = evidence.groupby('source').size()
starter_count = int(source_counts['starter'])
query_count = int(source_counts.drop('starter').sum())
assert len(evidence) == 279
assert starter_count == 175
assert query_count == 104
assert set(evidence['function']) == set(range(1, 9))
assert evidence.filter(regex=r'^x[1-8]$').stack().between(0, 1).all()
print({'starter_observations': starter_count, 'prospective_queries': query_count, 'total': len(evidence)})


## Starter evidence and participant-query results

The table deliberately separates the best course-supplied starter value from the best value achieved by the thirteen participant-selected queries. This prevents an early starter observation from being misreported as a later optimisation result.

In [ ]:
starter = evidence[evidence['source'].eq('starter')]
queries = evidence[~evidence['source'].eq('starter')].copy()
comparison = pd.DataFrame({
    'starter_best': starter.groupby('function')['output'].max(),
    'participant_query_best': queries.groupby('function')['output'].max(),
})
comparison['beat_starter_best'] = comparison['participant_query_best'] > comparison['starter_best']
comparison.index = [f'F{i}' for i in comparison.index]
comparison


In [ ]:
query_best = queries.groupby('function')['output'].max().sort_index().to_numpy()
recorded_best = winners['best_verified_output'].to_numpy()
assert len(query_best) == len(recorded_best) == 8
assert pd.Series(query_best).sub(recorded_best).abs().max() < 1e-12
print('PASS: final winner summary matches the strongest participant-query output for all eight functions.')


## Repeated-coordinate variability

Repeated coordinates are checked directly rather than assuming the portal response was deterministic.

In [ ]:
dimensions = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}
repeat_findings = []
for function, dimension in dimensions.items():
    coordinates = [f'x{i}' for i in range(1, dimension + 1)]
    frame = queries[queries['function'].eq(function)]
    groups = frame.groupby(coordinates, dropna=False)['output'].agg(['size', 'nunique', list]).reset_index()
    groups = groups[(groups['size'] > 1) & (groups['nunique'] > 1)].copy()
    if not groups.empty:
        groups.insert(0, 'function', f'F{function}')
        repeat_findings.append(groups)
repeatability = pd.concat(repeat_findings, ignore_index=True)
repeatability


In [ ]:
weekly = queries.assign(week=queries['source'].str.extract(r'(\d+)$')[0].astype(int))
fig, axes = plt.subplots(4, 2, figsize=(12, 14), constrained_layout=True)
for function, ax in zip(range(1, 9), axes.flat):
    frame = weekly[weekly['function'].eq(function)].sort_values('week')
    ax.plot(frame['week'], frame['output'], marker='o', color='#176B87')
    ax.set_title(f'Function {function}')
    ax.set_xlabel('Round')
    ax.set_ylabel('Objective output')
    ax.grid(alpha=0.2)
plt.suptitle('Observed participant-query trajectories across thirteen rounds', fontsize=15)
plt.show()


## Interpretation boundary

The trajectories describe only the submitted path through each bounded search space. They do not recover the hidden equations or prove global optima. F2, F3 and F6 returned different values at identical recorded coordinates, with the largest range in F6, so repeatability cannot be assumed across all functions. Post-capstone Black Box Resolution experiments are documented separately and do not alter these official results.